**Main analysis questions:**

1.   Are immunosenescenct immune states spatially correlated?
2.   Do dysfunctional cells and senescent cells cluster together?
3.   Does high immune infiltration correlate with senescence/dysfunction states?
4.   How do tissues 346 and 355 compare?



Note -- because these tissues are peritoneal lesions, I've subsetted only these tissue samples from the sc object to match the microenvironment.

_Future projects can compare signatures against other tissue types when they become available._

In [70]:
!pip install scanpy pandas numpy squidpy matplotlib anndata seaborn ptitprince

In [71]:
# -- Imports
import os
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import igraph
import seaborn as sns
import anndata as ad
import matplotlib.pyplot as plt
import ptitprince as pt
from matplotlib.patches import Patch

In [72]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [73]:
# -- Paths
input_path_sc_posterior = "/content/drive/MyDrive/endo-immune-atlas/data/processed/immune_reference_posterior.h5ad"

input_path_BEME_346 = "/content/drive/MyDrive/endo-immune-atlas/data/processed/BEME346_spatial_immune_architecture.h5ad"

input_path_BEME_355G = "/content/drive/MyDrive/endo-immune-atlas/data/processed/BEME355G_spatial_immune_architecture.h5ad"

sen_scores_path = "/content/drive/MyDrive/endo-immune-atlas/data/processed/06_senescence_scores.csv"

output_path_data = "/content/drive/MyDrive/endo-immune-atlas/data/processed"
output_path_figures = "/content/drive/MyDrive/endo-immune-atlas/results/final_figures"

os.makedirs(output_path_figures, exist_ok=True)

In [74]:
# -- COLOR PALETTES

spatial_palette  = [
    "#E69F00",
    "#00538A",
    "#BFBFBF",
    "#969696",
    "#009E73",
    "#6E6E6E",
    "#D55E00",
    "#A6BDD7",
    "#404040",
]

immune_palette = [
    "#D55E00",
    "#E69F00",
    "#8F8F8F",
    "#D9D9D9"
]

immune_palette_map = {
    "Immune-high": "#D55E00",
    "Immune-moderate": "#E69F00",
    "Immune-low": "#8F8F8F",
    "Immune-poor": "#D9D9D9"
}

In [75]:
# -- Import objects
immune_posterior = sc.read_h5ad(input_path_sc_posterior)
adata_346 = sc.read_h5ad(input_path_BEME_346)
adata_355G = sc.read_h5ad(input_path_BEME_355G)

I noticed last notebook that doing each code block for each tissue type is really time consuming and nonsensical. Going to start writing functions and code to reflect that from now on.

I plan to go back and clean up earlier notebooks with better functions to make the script run cleaner.

In [76]:
#============================================================================
# TISSUE DICT - update this when more tissues for analysis
#============================================================================
adatas = {
    "BEME346": adata_346,
    "BEME355G": adata_355G,
}

In [77]:
# -- CALCULCATE MEAN SCORES FOR SENESCENCE AND DYSFUNCTION PER TISSUE TYPE
mean_peri_sen_scores = (
    immune_posterior.obs
    .query("lesion_site == 'peritoneal'")
    .groupby("immune_cell_type", observed=True)["composite_sen_score"]
    .mean()
)

mean_peri_dys_scores = (
    immune_posterior.obs
    .query("lesion_site == 'peritoneal'")
    .groupby("immune_cell_type", observed=True)["dysfunction_score"]
    .mean()
)

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION_1 -- MAP SENESCENT AND DYSFUNCTION SCORES TO SPATIAL ARCHITECTURE
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def map_scores(adata, state_scores, state_col_name, abundance_col_name = "total_immune_spot_level"):
  """
  Maps a cell-type-level state score onto Visium spots using cell2location abundances.

  Parameters
  ---
  adata: AnnData
      Spatial AnnData object
  state_scores : pd.Series
      Indexed by immune cell type with average state scores
  state_col_name : str
      Name of output column
  abundance_col_name : str
      Column used for normalization
  """

  cell_types = state_scores.index.tolist()

  adata.obs[state_col_name] = (
      adata.obs[cell_types]
      .mul(state_scores, axis=1)
      .sum(axis=1)
      .div(adata.obs[abundance_col_name].replace(0, np.nan))
  )

  return adata

In [78]:
# -- MAP SENESCENT AND DYSFUNCTION SCORES TO SPATIAL ARCHITECTURE
for tissue, adata in adatas.items():
    map_scores(
        adata,
        mean_peri_sen_scores,
        "projected_senescence_score"
    )
    print(f"Finished senescence mapping for: {tissue}")

Finished senescence mapping for: BEME346
Finished senescence mapping for: BEME355G


In [79]:
# -- Order immune states for beautiful figures :)
immune_class_order = [
    "Immune-high",
    "Immune-moderate",
    "Immune-low",
    "Immune-poor"
]

In [81]:
#==============================================================
# FIGURE 9_1: SENESCENCE SCORES ON SPATIAL MAP
#==============================================================
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# matching scales
vmin = min(
    adata_346.obs["projected_senescence_score"].min(),
    adata_355G.obs["projected_senescence_score"].min()
)

vmax = max(
    adata_346.obs["projected_senescence_score"].max(),
    adata_355G.obs["projected_senescence_score"].max()
)

sq.pl.spatial_scatter(
    adata_346,
    color="projected_senescence_score",
    cmap="magma",
    size=1.5,
    vmin=vmin,
    vmax=vmax,
    colorbar=False,
    ax=ax[0],
    title="BEME346",
    img=False
)

sq.pl.spatial_scatter(
    adata_355G,
    color="projected_senescence_score",
    cmap="magma",
    size=1.5,
    vmin=vmin,
    vmax=vmax,
    colorbar=True,
    ax=ax[1],
    title="BEME355G",
    img=False
)

ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.09, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

plt.savefig(
    os.path.join(output_path_figures, "09_346_spatial_senescence_scores.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

Oddly, these senescent scores look dispersed in tissue 346 and clustered within immune poor regions in tissue 355G. Going to check correlation between abundance and projected senescence scores.

In [82]:
#=============================================================================
# FIGURE 9_2: CORRELATION BETWEEN PROJECTED SEN SCORES AND IMMUNE CLASS
#=============================================================================

fig, ax = plt.subplots(1, 2, figsize=(10,5), sharey=True, constrained_layout=True)

sns.scatterplot(
    data=adata_346.obs,
    x="total_immune_spot_level",
    y="projected_senescence_score",
    alpha=0.7,
    hue="tissue_level_immune_class",
    legend=False,
    hue_order=immune_class_order,
    palette=immune_palette,
    ax=ax[0]
)

sns.regplot(
    data=adata_346.obs,
    x="total_immune_spot_level",
    y="projected_senescence_score",
    scatter=False,
    lowess=True,
    color="black",
    line_kws={"lw":3},
    ax=ax[0]
)


sns.scatterplot(
    data=adata_355G.obs,
    x="total_immune_spot_level",
    y="projected_senescence_score",
    alpha=0.7,
    hue="tissue_level_immune_class",
    hue_order=immune_class_order,
    palette=immune_palette,
    ax=ax[1]
)


sns.regplot(
    data=adata_355G.obs,
    x="total_immune_spot_level",
    y="projected_senescence_score",
    lowess=True,
    scatter=False,
    color="black",
    line_kws={"lw":3},
    ax=ax[1]
)

# -- Titles
ax[0].set_title("BEME346", fontsize=15)
ax[1].set_title("BEME355G", fontsize=15)

# X labels
ax[0].set_xlabel("Total Immune Abundance")
ax[1].set_xlabel("Total Immune Abundance")

# Y labels
ax[0].set_ylabel("Projected Senescence Score")
ax[1].legend(title=None)


# Figure labels
ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)


plt.savefig(
    os.path.join(output_path_figures, "09_immune_abundance_sen_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [83]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION_2 -- CALCULATE CLUSTER BASED MEANS AND MAP BACK TO OBJECT
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def map_cluster_means(adata, cluster_col, mean_col_name, score_col):
  """
  Calculates the desired score mean and then maps that score back to the object.


  Parameters
  ---
  adata: AnnData
      Spatial AnnData object
  cluster_col : str
      Column containing spatial cluster labels.
  mean_col_name : str
      Name of the new column created
  score_col : str
      Column of scores to average
  """

  cluster_means = (
      adata.obs
      .groupby(cluster_col, observed=True)[score_col]
      .mean()
  )

  adata.obs[mean_col_name] = (
      adata.obs[cluster_col].map(cluster_means)
  )

  return cluster_means

In [84]:
# -- CALCULATE CLUSTER MEANS
cluster_means = {}

for tissue, adata in adatas.items():
    cluster_means[tissue] = map_cluster_means(
        adata,
        cluster_col="leiden_0.4",
        mean_col_name="cluster_mean_senescence",
        score_col="projected_senescence_score",
    )

    print(f"Finished cluster means mapping for: {tissue}")

Finished cluster means mapping for: BEME346
Finished cluster means mapping for: BEME355G


In [85]:
#==============================================================
# FIGURE 9_3: SENESCENT SCORES BY CLUSTER REGIONS
#==============================================================

# creating summary to map immune states to bar colors

# make this into a function -- "cluster_summary"
cluster_summary_346 = (
    adata_346.obs
    .groupby("leiden_0.4", observed=True)
    .agg(
        mean_senescence=("projected_senescence_score", "mean"),
        immune_class=("tissue_level_immune_class", "first")
    )
    .reset_index()
)

cluster_summary_355G = (
    adata_355G.obs
    .groupby("leiden_0.4", observed=True)
    .agg(
        mean_senescence=("projected_senescence_score", "mean"),
        immune_class=("tissue_level_immune_class", "first")
    )
    .reset_index()
)

cluster_summary_346["color"] = (
    cluster_summary_346["immune_class"]
    .map(immune_palette_map)
)

cluster_summary_355G["color"] = (
    cluster_summary_355G["immune_class"]
    .map(immune_palette_map)
)



fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# 346
ax[0].bar(
    cluster_summary_346["leiden_0.4"].astype(str),
    cluster_summary_346["mean_senescence"],
    color=cluster_summary_346["color"]
)
# 355G
ax[1].bar(
    cluster_summary_355G["leiden_0.4"].astype(str),
    cluster_summary_355G["mean_senescence"],
    color=cluster_summary_355G["color"]
)

ax[0].set_title("BEME346")
ax[1].set_title("BEME355G")

ax[0].set_xlabel("Spatial cluster")
ax[1].set_xlabel("Spatial cluster")

ax[0].set_ylabel("Mean projected senescence")
ax[1].set_ylabel("")


ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

# add legend
legend_handles = [
    Patch(color=immune_palette_map["Immune-high"], label="Immune-high"),
    Patch(color=immune_palette_map["Immune-moderate"], label="Immune-moderate"),
    Patch(color=immune_palette_map["Immune-low"], label="Immune-low"),
    Patch(color=immune_palette_map["Immune-poor"], label="Immune-poor"),
]

ax[1].legend(
    handles=legend_handles,
    frameon=True,
)


plt.savefig(
    os.path.join(output_path_figures, "09_spatial_regions_sen_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [86]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION_3 -- MAKE DF NEEDED FOR PLOTS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def make_plot_df(
    adata,
    columns,
    tissue_name=None,
):
    """
    Create a plotting dataframe from an AnnData object.

    Parameters
    ----------
    adata : AnnData
    columns : list[str]
        Columns from adata.obs to keep.
    tissue_name : str, optional
        Adds a 'tissue' column if provided.

    Returns
    -------
    pd.DataFrame
    """
    df = adata.obs[columns].copy()

    if tissue_name is not None:
        df["tissue"] = tissue_name

    return df

In [87]:
#==============================================================
# FIGURE 9_4: DISTRIBUTION OF SCORES ACROSS IMMUNE CLASSES
#==============================================================
rain_plot_df_346 = make_plot_df(adata_346,
                                columns=[
                                    "projected_senescence_score",
                                    "tissue_level_immune_class",
                                    "leiden_0.4"],
                                tissue_name="346"
                                )

rain_plot_df_355G = make_plot_df(adata_355G,
                                columns=[
                                    "projected_senescence_score",
                                    "tissue_level_immune_class",
                                    "leiden_0.4"],
                                tissue_name="355G"
                                )

for df in [rain_plot_df_346, rain_plot_df_355G]:
    df["tissue_level_immune_class"] = pd.Categorical(
        df["tissue_level_immune_class"],
        categories=immune_class_order,
        ordered=True
    )

fig, ax = plt.subplots(1, 2, figsize=(10,5), sharey=True, constrained_layout=True)

pt.RainCloud(
    data=rain_plot_df_346,
    x="tissue_level_immune_class",
    y="projected_senescence_score",
    width_viol= 1,
    width_box=0.4,
    hue="tissue_level_immune_class",
    palette=immune_palette,
    orient="h",
    ax=ax[0]
)

pt.RainCloud(
    data=rain_plot_df_355G,
    x="tissue_level_immune_class",
    y="projected_senescence_score",
    width_viol= 1,
    width_box=0.4,
    hue="tissue_level_immune_class",
    palette=immune_palette,
    orient="h",
    ax=ax[1]
)


ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[0].set_title("BEME346", fontsize=15)
ax[1].set_title("BEME355G", fontsize=15)
ax[0].set_xlabel("Projected Senescence Score")
ax[0].set_ylabel("Immune Class")
ax[0].set_title("Tissue 346", fontsize=16)


ax[1].set_xlabel("Projected Senescence Score")
ax[1].set_ylabel("")
for a in ax:
    a.set_xscale("log")
    a.set_xlim(0.0001, 0.2)
ax[1].set_title("Tissue 355G", fontsize=16)

plt.savefig(
    os.path.join(output_path_figures, "09_immune_class_sen_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):


# **DYSFUNCTION SECTION**

In [88]:
# -- MAP SENESCENT AND DYSFUNCTION SCORES TO SPATIAL ARCHITECTURE
for tissue, adata in adatas.items():
    map_scores(
        adata,
        mean_peri_dys_scores,
        "projected_dysfunction_score"
    )
    print(f"Finished dysfunction mapping for: {tissue}")

Finished dysfunction mapping for: BEME346
Finished dysfunction mapping for: BEME355G


In [89]:
#==============================================================
# FIGURE 9_5: SENESCENCE SCORES ON SPATIAL MAP
#==============================================================
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# matching scales
vmin = min(
    adata_346.obs["projected_dysfunction_score"].min(),
    adata_355G.obs["projected_dysfunction_score"].min()
)

vmax = max(
    adata_346.obs["projected_dysfunction_score"].max(),
    adata_355G.obs["projected_dysfunction_score"].max()
)

sq.pl.spatial_scatter(
    adata_346,
    color="projected_dysfunction_score",
    cmap="magma",
    size=1.5,
    vmin=vmin,
    vmax=vmax,
    colorbar=False,
    ax=ax[0],
    title="BEME346",
)

sq.pl.spatial_scatter(
    adata_355G,
    color="projected_dysfunction_score",
    cmap="magma",
    size=1.5,
    vmin=vmin,
    vmax=vmax,
    colorbar=True,
    ax=ax[1],
    title="BEME355G",
)

ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.09, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

plt.savefig(
    os.path.join(output_path_figures, "09_355G_spatial_dysfunction_scores.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [90]:
#=============================================================================
# FIGURE 9_6: CORRELATION BETWEEN PROJECTED SEN SCORES AND IMMUNE CLASS
#=============================================================================

fig, ax = plt.subplots(1, 2, figsize=(10,5), sharey=True, constrained_layout=True)

sns.scatterplot(
    data=adata_346.obs,
    x="total_immune_spot_level",
    y="projected_dysfunction_score",
    alpha=0.7,
    hue="tissue_level_immune_class",
    legend=False,
    hue_order=immune_class_order,
    palette=immune_palette,
    ax=ax[0]
)

sns.regplot(
    data=adata_346.obs,
    x="total_immune_spot_level",
    y="projected_dysfunction_score",
    scatter=False,
    lowess=True,
    color="black",
    line_kws={"lw":3},
    ax=ax[0]
)


sns.scatterplot(
    data=adata_355G.obs,
    x="total_immune_spot_level",
    y="projected_dysfunction_score",
    alpha=0.7,
    hue="tissue_level_immune_class",
    hue_order=immune_class_order,
    palette=immune_palette,
    ax=ax[1]
)


sns.regplot(
    data=adata_355G.obs,
    x="total_immune_spot_level",
    y="projected_dysfunction_score",
    lowess=True,
    scatter=False,
    color="black",
    line_kws={"lw":3},
    ax=ax[1]
)

# -- Titles
ax[0].set_title("BEME346", fontsize=15)
ax[1].set_title("BEME355G", fontsize=15)

# X labels
ax[0].set_xlabel("Total Immune Abundance")
ax[1].set_xlabel("Total Immune Abundance")

# Y labels
ax[0].set_ylabel("Projected Dysfunction Score")
ax[1].legend(title=None)


# Figure labels
ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

plt.savefig(
    os.path.join(output_path_figures, "09_immune_abundance_dysfunction_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [91]:
# -- CALCULATE CLUSTER MEANS
cluster_means = {}

for tissue, adata in adatas.items():
    cluster_means[tissue] = map_cluster_means(
        adata,
        cluster_col="leiden_0.4",
        mean_col_name="cluster_mean_senescence",
        score_col="projected_dysfunction_score",
    )

    print(f"Finished cluster means mapping for: {tissue}")

Finished cluster means mapping for: BEME346
Finished cluster means mapping for: BEME355G


In [92]:
#==============================================================
# FIGURE 9_7: SENESCENT SCORES BY CLUSTER REGIONS
#==============================================================

# creating summary to map immune states to bar colors

# make this into a function -- "cluster_summary"
cluster_summary_346 = (
    adata_346.obs
    .groupby("leiden_0.4", observed=True)
    .agg(
        mean_dysfunction=("projected_dysfunction_score", "mean"),
        immune_class=("tissue_level_immune_class", "first")
    )
    .reset_index()
)

cluster_summary_355G = (
    adata_355G.obs
    .groupby("leiden_0.4", observed=True)
    .agg(
        mean_dysfunction=("projected_dysfunction_score", "mean"),
        immune_class=("tissue_level_immune_class", "first")
    )
    .reset_index()
)

cluster_summary_346["color"] = (
    cluster_summary_346["immune_class"]
    .map(immune_palette_map)
)

cluster_summary_355G["color"] = (
    cluster_summary_355G["immune_class"]
    .map(immune_palette_map)
)



fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# 346
ax[0].bar(
    cluster_summary_346["leiden_0.4"].astype(str),
    cluster_summary_346["mean_dysfunction"],
    color=cluster_summary_346["color"]
)
# 355G
ax[1].bar(
    cluster_summary_355G["leiden_0.4"].astype(str),
    cluster_summary_355G["mean_dysfunction"],
    color=cluster_summary_355G["color"]
)

ax[0].set_title("BEME346")
ax[1].set_title("BEME355G")

ax[0].set_xlabel("Spatial cluster")
ax[1].set_xlabel("Spatial cluster")

ax[0].set_ylabel("Mean projected dysfunction")
ax[1].set_ylabel("")


ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

# add legend
legend_handles = [
    Patch(color=immune_palette_map["Immune-high"], label="Immune-high"),
    Patch(color=immune_palette_map["Immune-moderate"], label="Immune-moderate"),
    Patch(color=immune_palette_map["Immune-low"], label="Immune-low"),
    Patch(color=immune_palette_map["Immune-poor"], label="Immune-poor"),
]

ax[1].legend(
    handles=legend_handles,
    frameon=True,
)


plt.savefig(
    os.path.join(output_path_figures, "09_spatial_regions_dysfunction_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [93]:
#==============================================================
# FIGURE 9_8: DISTRIBUTION OF SCORES ACROSS IMMUNE CLASSES
#==============================================================
rain_plot_df_346 = make_plot_df(adata_346,
                                columns=[
                                    "projected_dysfunction_score",
                                    "tissue_level_immune_class",
                                    "leiden_0.4"],
                                tissue_name="346"
                                )

rain_plot_df_355G = make_plot_df(adata_355G,
                                columns=[
                                    "projected_dysfunction_score",
                                    "tissue_level_immune_class",
                                    "leiden_0.4"],
                                tissue_name="355G"
                                )

for df in [rain_plot_df_346, rain_plot_df_355G]:
    df["tissue_level_immune_class"] = pd.Categorical(
        df["tissue_level_immune_class"],
        categories=immune_class_order,
        ordered=True
    )

fig, ax = plt.subplots(1, 2, figsize=(10,5), sharey=True, constrained_layout=True)

pt.RainCloud(
    data=rain_plot_df_346,
    x="tissue_level_immune_class",
    y="projected_dysfunction_score",
    width_viol= 1,
    width_box=0.4,
    hue="tissue_level_immune_class",
    palette=immune_palette,
    orient="h",
    ax=ax[0]
)

pt.RainCloud(
    data=rain_plot_df_355G,
    x="tissue_level_immune_class",
    y="projected_dysfunction_score",
    width_viol= 1,
    width_box=0.4,
    hue="tissue_level_immune_class",
    palette=immune_palette,
    orient="h",
    ax=ax[1]
)


ax[0].text(
    0, 1.03, "A",
    transform=ax[0].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[1].text(
    0, 1.03, "B",
    transform=ax[1].transAxes,
    fontsize=18,
    fontweight="bold"
)

ax[0].set_title("BEME346", fontsize=15)
ax[1].set_title("BEME355G", fontsize=15)
ax[0].set_xlabel("Projected Dysfunction Score")
ax[0].set_ylabel("Immune Class")
ax[0].set_title("Tissue 346", fontsize=16)


ax[1].set_xlabel("Projected Dysfunction Score")
ax[1].set_ylabel("")
for a in ax:
    a.set_xscale("log")
    a.set_xlim(0.0001, 0.2)
ax[1].set_title("Tissue 355G", fontsize=16)


plt.savefig(
    os.path.join(output_path_figures, "09_immune_class_dysfunction_score.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):


In [94]:
# -- SAVE DATA
adata_346.write_h5ad(
    os.path.join(output_path_data, "BEME346_spatial_immunosenescence.h5ad")
)

adata_355G.write_h5ad(
    os.path.join(output_path_data, "BEME355G_spatial_immunosenescence.h5ad")
)

Notebook Overview

The senescence scores from scRNA-seq analysis were mapped onto the spatial maps of peritoneal lesions using a linear mixture model.

Key results
1. Projected senescence scores do not map cleanly to immune states as initially hypothesized. I assumed that there would be higher senescence in immune rich areas. I think this could be because I am only assessing immune based senescence but tissues are a mix between cells - non-immune cells (stromal, epitelial, etc.) cells may also be driving the senescence in the spots on the tissues.

2. There is less distribution in projected senescence & dysfunction scores in tissue 355G. Interestingly, this patient is older and in a different menstrual phase than the other patient. -- Not making heavy conclusions but this is an interesting insight.

3. Dysfunction scores have similar distributions to senescence scores, but map differently to clusters in tissue 346, suggesting that in some regions, senescence and dysfunction present differently.  



Future work
1. Address immune/non-immune proportions by normalizing the amount of immune cells/spot then assessing senescent signatures.